In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/q03_rewrite/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Filter and drop unused columns early
define_date = '1995-03-04'

fcustomer = customer.query("C_MKTSEGMENT == 'HOUSEHOLD'")[['C_CUSTKEY']]
forders  = orders.query(f"O_ORDERDATE < '{define_date}'")[['O_ORDERKEY','O_CUSTKEY','O_ORDERDATE','O_SHIPPRIORITY']]
flineitem = lineitem.query(f"L_SHIPDATE > '{define_date}'")[['L_ORDERKEY','L_EXTENDEDPRICE','L_DISCOUNT']]

# Join on the correct key names
jn2 = (
    fcustomer
      .merge(forders, left_on='C_CUSTKEY', right_on='O_CUSTKEY')
      .merge(flineitem, left_on='O_ORDERKEY', right_on='L_ORDERKEY')
)

# Compute TMP, aggregate and sort
total = (
    jn2
      .assign(TMP = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT))
      .groupby(
          ['L_ORDERKEY','O_ORDERDATE','O_SHIPPRIORITY'],
          as_index=False,
          sort=False
      )
      .agg({'TMP':'sum'})
      .sort_values('TMP', ascending=False)
)

# Final projection
res = total[['L_ORDERKEY','TMP','O_ORDERDATE','O_SHIPPRIORITY']]


CPU times: user 40 s, sys: 7.25 s, total: 47.3 s
Wall time: 47.3 s


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_1.pickle